# 05 — Active Sampling / Closed-Loop Candidate Proposal

최적화가 미비하거나 uncertainty가 큰 경우, 대규모 virtual generator pool에서 **성능 접근성 + surrogate uncertainty + descriptor novelty**를 함께 평가하고 diverse batch를 선택합니다.


In [ ]:
from pathlib import Path
import sys,joblib
PROJECT_POINTER=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation\.ai_voxel_ml_project.json")
CODE_DIR=Path.cwd()/"Code" if (Path.cwd()/"Code").exists() else Path.cwd();sys.path.insert(0,str(CODE_DIR)) if str(CODE_DIR) not in sys.path else None
N_VIRTUAL=4096;N_SELECT=18;RANDOM_SEED=4242
TARGET_SPEC={
 'plateau_stress':{'mode':'target','target':5.0,'scale':2.0,'weight':1.0},
 'densification_strain':{'mode':'maximize','min':0.45,'scale':0.20,'weight':1.0},
 'absorbed_energy_to_densification':{'mode':'maximize','scale':5.0,'weight':1.2},
 'peak_stress':{'mode':'minimize','max':10.0,'scale':4.0,'weight':0.6},
}
ACQUISITION_WEIGHTS={'performance':0.40,'uncertainty':0.35,'novelty':0.25}


In [ ]:
import joblib
from voxel_ml_common import load_contract,mark_stage
from active_learning_engine import propose_active_samples
c=load_contract(PROJECT_POINTER);mr=Path(c['model_root']);out=Path(c['active_root']);out.mkdir(parents=True,exist_ok=True)
gen=joblib.load(mr/'01_gen2desc'/'gen2desc_bundle.joblib');curve=joblib.load(mr/'02_desc2curve'/'desc2curve_bundle.joblib')
q=propose_active_samples(gen,curve,TARGET_SPEC,out,N_VIRTUAL,N_SELECT,RANDOM_SEED,ACQUISITION_WEIGHTS)
mark_stage(out,'01_active_sampling','completed',[out/'active_sampling_generator_parameters.csv'],{'n_virtual':N_VIRTUAL,'n_select':len(q)})
display(q);print('\nDatasetFactory input:',out/'active_sampling_generator_parameters.csv')


### Closed loop
새 RUN_NAME에서 DatasetFactory를 `CANDIDATE_SOURCE="active_sampling"`으로 실행 → Voxel/DLP 제작 → 압축시험 → Notebook 04 ingest → Notebook 01/02 retrain → Notebook 03 optimization을 반복합니다.
